# 🚀 ViSceT5 — **PreSTU SplitOCR + MRA** Pretrain → Finetune · Colab **A100**

Pretrain **PreSTU SplitOCR** (đúng paper): sắp OCR theo thứ tự đọc, cắt ngẫu nhiên → prompt chứa phần đầu, mô hình **sinh phần còn lại từ ảnh** (prefix/target rời nhau, buộc ĐỌC pixel; `full_ocr_prob=0.2` thỉnh thoảng sinh toàn bộ). Không bbox/ground. Kèm **MRA** đúng Feast-Your-Eyes: ViT-base-patch16 **336** (nội suy 2D bicubic từ 224) + ConvNeXt-V2-Base **1024** (đặc trưng CUỐI của CNN bơm vào **3 stage cuối** ViT), VS TẮT, `VISION_LR_SCALE=0.2`.

Chống overfit (dữ liệu ít): **cosine LR + 6 epoch**. Sau pretrain → finetune ViTextVQA giữ MRA.

Nhánh `exp/mra-pretrain`. Runtime → **A100**. Internet ON.

## 1. Clone Codebase & Checkout Nhánh Pretrain
Tự động phát hiện môi trường (Kaggle hoặc Colab), đồng bộ repository từ GitHub và chuyển sang nhánh `exp/pretrain-gen-all` chứa các cải tiến mới nhất.

In [ ]:
import os
import sys

# Tự động phát hiện thư mục làm việc (Kaggle: /kaggle/working | Colab: /content)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout exp/mra-pretrain
!git pull origin exp/mra-pretrain
!git log --oneline -3

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Cài Đặt Môi Trường Chuẩn (Transformers 4.45.2 Cố Định)
Gỡ các phiên bản thư viện mặc định của Kaggle và cài đặt chính xác các phiên bản tương thích từ `requirements.txt`.

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap
!pip install -q --upgrade --no-cache-dir gdown

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

## 3. Chuẩn Bị Dữ Liệu Tiền Huấn Luyện (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache vào `output/pretrain`.

In [ ]:
# Định vị thư mục lưu dataset CSV đồng bộ với visualize và trainer
%env OUTPUT_PATH=./output/pretrain
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

### 3b. Xem thử vài mẫu SplitOCR (kiểm tra prefix/target trước khi train)

In [ ]:
# === Xem thử vài mẫu SplitOCR (prefix vs target) để kiểm tra bằng mắt ===
# VinText: TARGET = GT (labels, sạch), PREFIX = text SwinTextSpotter (silver, nhiễu), box rời nhau.
# EVJVQA: không có GT -> tách trên spotter như bình thường.
import os, random, numpy as np, pandas as pd, torch
from PIL import Image
from data.gt_ocr import load_vintext_gt
from data.collator import (_splitocr_b1_gt, _split_ocr_sequential,
                           _sort_ocr_reading_order, _normalize_text)

OUT = os.environ.get("OUTPUT_PATH", "./output/pretrain")
df = pd.read_csv(os.path.join(OUT, "merged_train.csv"))
print("columns:", list(df.columns))
print("has label_path:", "label_path" in df.columns,
      "| #VinText có GT:", int(df["label_path"].notna().sum()) if "label_path" in df.columns else 0)
random.seed(0)

def show(row, tag):
    ip, op = row["image_path"], row["ocr_path"]
    lp = row["label_path"] if "label_path" in row and pd.notna(row["label_path"]) else None
    W, H = Image.open(ip).convert("RGB").size
    sp = np.load(op, allow_pickle=True).item()
    sp_boxes = torch.tensor(np.asarray(sp.get("boxes")), dtype=torch.float)
    sp_texts = list(sp.get("texts", []))
    if isinstance(lp, str) and os.path.exists(lp):
        gt_t, gt_b = load_vintext_gt(lp, W, H)
        gt_t, gt_b = _sort_ocr_reading_order(gt_t, gt_b)
        pfx, tgt = _splitocr_b1_gt(gt_t, gt_b, sp_boxes, sp_texts, full_ocr_prob=0.2)
        src = "VinText  (PREFIX=spotter SILVER, TARGET=GT sạch)"
    else:
        toks = [_normalize_text(t, lowercase=True).strip() for t in sp_texts
                if isinstance(t, str) and t.strip()]
        bx = sp_boxes[:len(toks)] if sp_boxes.size(0) >= len(toks) else torch.zeros(len(toks), 4)
        toks, bx = _sort_ocr_reading_order(toks, bx)
        pw, _, tw, _ = _split_ocr_sequential(toks, bx, full_ocr_prob=0.2)
        pfx, tgt = " ".join(pw), " ".join(tw)
        src = "EVJVQA   (spotter-only split)"
    print("=" * 78)
    print(f"[{tag}] {os.path.basename(str(ip))} | {src}")
    print("  PROMPT :", ("Generate ocr_text in vi: " + pfx).strip() if pfx else "Generate ocr_text in vi:")
    print("  TARGET :", tgt[:220])

vin = df[df["label_path"].notna()] if "label_path" in df.columns else df.iloc[0:0]
evj = df[df["dataset"].astype(str).str.upper().str.contains("EVJ")] if "dataset" in df.columns else df.iloc[0:0]
for _, r in vin.sample(min(3, len(vin))).iterrows(): show(r, "VinText")
for _, r in evj.sample(min(2, len(evj))).iterrows(): show(r, "EVJVQA")
print("\n✅ Kiểm tra: VinText TARGET phải là GT có dấu/đúng hoa-thường; PREFIX là bản spotter (thường sai/thiếu).")


## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)
Khởi tạo `OpenViVQAModel`, tải các trọng số nền tảng ViT5 và CLIP-ViT, kiểm tra tính toàn vẹn số học.

In [ ]:
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

### Cấu hình tối ưu toàn diện:
* **Epochs:** 10
* **Batch size:** 4 (per device) $\times$ 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** $1\times 10^{-4}$ (ViT5) và $1\times 10^{-5}$ (CLIP ViT unfrozen 4 layers)
* **Loss balance:** $\lambda_{\text{bbox}} = 0.3$
* **Vision Unfreeze:** Top-4 layers (`vision_unfreeze_last_n = 4`) + post-layernorm
* **Target Split:** Spatial Region Clustering (Khoanh vùng cụm không gian)
* **Output dir:** `/kaggle/working/pretrain_output`

### 5a. SMOKE test PreSTU+MRA (vài step) — bắt lỗi wiring trước khi chạy full

In [ ]:
# SMOKE: vài step, bắt lỗi wiring PreSTU SplitOCR + MRA trước khi chạy full.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n 4 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --smoke_test True \
    --output_dir ./output/pretrain_mra_smoke \
    --logging_dir ./output/pretrain_mra_smoke/logs


### 5b. FULL PreSTU SplitOCR + MRA pretrain (10 epoch)

In [ ]:
# PreSTU SplitOCR (ĐÚNG paper): sinh phần OCR text còn lại từ ảnh (prefix->target rời nhau).
# split_mode=sequential, full_ocr_prob=0.2 (curriculum độ dài target — hợp dữ liệu ít).
# Chống overfit: cosine LR decay + 6 epoch. MRA: ViT-base-patch16 336 + ConvNeXt-V2-Base 1024 (đặc trưng CUỐI -> 3 stage cuối ViT), VS TẮT.
# Tiết kiệm VRAM chống OOM: gradient_checkpointing True + bf16 True + per_device_train_batch_size 2 x accum 8 (effective batch size 16).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n 4 \
    --num_train_epochs 6 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.05 \
    --learning_rate 0.0001 \
    --weight_decay 0.01 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 8 \
    --save_total_limit 2 \
    --output_dir ./output/pretrain_mra \
    --logging_dir ./output/pretrain_mra/logs


## 6. Trực Quan Hóa Kết Quả & Attention Heatmap (Interactive Visual Inspection)

Cell này tải checkpoint tốt nhất vừa huấn luyện và trực quan hóa 3 khung hình song song:
1. **Khung 1 (Ground-Truth):** Ảnh gốc + Hộp BBox Tiền tố (Xanh dương) + Hộp BBox Hậu tố Mục tiêu trong vùng khoanh (Xanh lá) + Chuỗi từ Suffix chuẩn.
2. **Khung 2 (Model Prediction):** Ảnh gốc + Hộp BBox Hậu tố mô hình dự đoán (Đỏ) + Chuỗi từ Suffix do ViT5 Decoder sinh ra qua Beam Search.
3. **Khung 3 (Visual Focus Attention Heatmap):** Bản đồ nhiệt chú ý không gian của Visual Search (AVF) đè lên ảnh gốc, chỉ rõ vùng mắt mô hình đang tập trung nhìn khi sinh từ vựng.

In [ ]:
# ⚠️ Trực quan hóa Attention Heatmap dùng AVF (Visual Search) — MRA chạy VS TẮT nên bỏ qua khung heatmap.
# Nếu muốn xem bbox/суffix dự đoán, chạy visualize với model VS-on riêng. (bỏ qua ở pipeline MRA)
print('Bỏ qua visualize AVF heatmap ở chế độ MRA (VS off).')

## 7. Chuyển Giao Sang Downstream VQA (Transfer Learning to Fine-Tuning)

Sau khi tiền huấn luyện hoàn tất, mô hình đã sẵn sàng chuyển giao tri thức sang bài toán Scene-Text VQA tiếng Việt (**ViTextVQA**).
Toàn bộ trọng số của `vit5.encoder`, `vit5.decoder`, `qa_clip.vision_model` (4 lớp thích ứng) và `visual_search` được nạp nguyên vẹn vào `training/finetune.py`.

In [ ]:
# Finetune downstream ViTextVQA, warm-start từ checkpoint PreSTU+MRA, GIỮ MRA (VS off, 768).
# bf16 + TF32 + gradient_checkpointing; batch 2 x accum 4 = effective 8 (chống OOM tuyệt đối).
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python training/finetune.py configs/finetune.yaml \
    --dataset_name "ViTextVQA" \
    --clip_vision_name openai/clip-vit-base-patch16 \
    --clip_image_size 336 \
    --vs_backbone facebook/convnextv2-base-22k-384 \
    --model_name_or_path ./output/pretrain_mra \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 2 \
    --per_device_eval_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 0.00003 \
    --gradient_checkpointing True \
    --bf16 True \
    --tf32 True \
    --output_dir ./output/finetune_mra \
    --logging_dir ./output/finetune_mra/logs


## 8. Lưu / Tải checkpoint pretrain (Colab zip-download hoặc HuggingFace)

In [ ]:
# === Tải checkpoint PRETRAIN về máy (Colab) ===
# output_dir của pretrain là ./output/pretrain_mra (KHÔNG phải /kaggle/...).
import os, shutil
CKPT = "output/pretrain_mra"
assert os.path.isdir(CKPT), f"Không thấy {CKPT} — chạy cell pretrain (5b) trước."

# Bản NHẸ để warm-start finetune: chỉ model.safetensors + config + tokenizer (bỏ optimizer/rng).
LIGHT = "pretrain_mra_light"
if os.path.isdir(LIGHT): shutil.rmtree(LIGHT)
os.makedirs(LIGHT, exist_ok=True)
KEEP = ("model.safetensors", "config.json", "generation_config.json",
        "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
        "spiece.model", "added_tokens.json")
for f in KEEP:
    src = os.path.join(CKPT, f)
    if os.path.exists(src): shutil.copy2(src, os.path.join(LIGHT, f))
print("Files (light):", os.listdir(LIGHT))
zip_path = shutil.make_archive("ViSceT5_PreSTU_MRA_pretrain", "zip", LIGHT)
print("Đã nén:", os.path.abspath(zip_path), "| size:", round(os.path.getsize(zip_path)/1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)     # trình duyệt tự tải về
except Exception as e:
    print("Không tự tải (không ở Colab UI?). Tải thủ công ở panel Files bên trái:", zip_path)

In [ ]:
# === (Tùy chọn) Đẩy checkpoint pretrain lên HuggingFace để dùng lại sau ===
from huggingface_hub import HfApi
HF_TOKEN = ""                       # <== điền token HF (write)
HF_REPO  = ""                       # vd: "Kus669/ViSceT5-mra-pretrain"
if HF_TOKEN and HF_REPO:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path="output/pretrain_mra", repo_id=HF_REPO, repo_type="model",
                      ignore_patterns=["*optimizer.pt", "*rng_state*", "*scheduler.pt", "checkpoint-*/*"])
    print("✅ Uploaded -> https://huggingface.co/%s" % HF_REPO)
    print("   Finetune từ nơi khác: --model_name_or_path %s (tải bằng snapshot_download)" % HF_REPO)
else:
    print("Điền HF_TOKEN + HF_REPO rồi chạy lại để upload.")